In [ ]:
import pandas as pd
import numpy as np

# PPMI

## Preprocessing

### Prepare metadata

In [ ]:
from create_master_metadata import create_master_metadata

master_metadata = create_master_metadata(
    rnaseq_meta_path='../../metadata.csv',
    proteomics_meta_path='../../proteomics/data/proteomics_metadata.csv',
    output_path='../data/metadata_m0_PPMI.csv',
    cohort="PPMI"
)

In [ ]:
# Checkpoint
meta_ppmi = pd.read_csv('../data/metadata_m0_PPMI.csv')

### Prepare expression data

### Preprocess RNAseq data

In [ ]:
from preprocess_rnaseq import preprocess_rnaseq

In [ ]:
log_matrix = preprocess_rnaseq(
    counts_file="../data/ppmi_rnaseq.csv",
    metadata_file="../data/master_baseline_metadata.csv",
    output_file="../data/ppmi_rnaseq_log1p_filtered.csv",
    visit_filter=0.0,
    cohort_name="PPMI"
)

#### QC checks for preprocessing step

In [ ]:
# Load your generated log1p matrix
log_df = pd.read_csv("../data/ppmi_rnaseq_log1p_filtered.csv", index_col=0)

In [ ]:
# Check for NaNs or infinities
print("Any NaN values?", log_df.isna().sum().sum())
print("Any Infinite values?", np.isinf(log_df.select_dtypes(include=np.number)).sum().sum())
print(f"Matrix shape (Genes x Samples): {log_df.shape}")

In [ ]:
print(log_df.iloc[:5, :5]) # Preview top-left corner
print("Overall Mean:", log_df.values.mean())
print("Overall Std Dev:", log_df.values.std())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# Transpose so samples are rows for PCA (Samples x Genes)
pca_data = log_df.T

pca = PCA(n_components=5)
pcs = pca.fit_transform(pca_data)
explained_var = pca.explained_variance_ratio_ * 100
print(f"PC1 ({explained_var[0]:.1f}%)", f"PC2 ({explained_var[1]:.1f}%)")

pca_df = pd.DataFrame(pcs, index=pca_data.index)

# Merge with metadata to color-code by technical factors (e.g., plate)
m = meta.loc[meta['study']=='PPMI'].set_index('participant_id')[['plate', 'sex', 'diagnosis']]

pca_df = pca_df.join(m, how='inner')

# Plot PCA colored by sequencing plate
plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_df, x=pca_df.columns[0], y=pca_df.columns[1], hue='plate', alpha=0.7, palette='tab10', legend=False)
plt.title("PCA of Log1p Matrix (Pre-Batch Correction, colored by plate)")
plt.xlabel(f"PC1 ({explained_var[0]:.1f}%)")
plt.ylabel(f"PC2 ({explained_var[1]:.1f}%)")
#plt.savefig( "../figures/ppmi_log1p_preBatch_byPlate_PCA1.jpg", dpi=600,bbox_inches="tight")
plt.show()


### RNA-seq technical batch correction

In [ ]:
! ./adjust_batch_effects.R \
--expression ../data/ppmi_rnaseq_log1p_filtered.csv \
--metadata ../data/master_baseline_metadata.csv \
--output ../data/ppmi_rnaseq_adjusted.csv \
--plot_prefix ../figures/ppmi_diagnostic_plots

### Proteomics

In [ ]:
from proteomics_qc import verify_proteomics_scaling, normalize_proteomics

In [ ]:
csfp_df = pd.read_csv("../data/ppmi_csf_prot.csv", index_col=0)
plap_df = pd.read_csv("../data/ppmi_pla_prot.csv", index_col=0)

In [ ]:
csf_vars = verify_proteomics_scaling(csfp_df, "CSF Proteomics")

In [ ]:
plasma_vars = verify_proteomics_scaling(plap_df, "Plasma Proteomics")

In [ ]:
# Apply normalization (log2 + median centering)
csfp_clean = normalize_proteomics(csfp_df, log2_transform=True)
plap_clean = normalize_proteomics(plap_df, log2_transform=True)

## MOFA training

### MOFA prep

In [ ]:
from prep_utils import validate_and_intersect_views, run_preflight_check

In [ ]:
aligned_views, aligned_meta, mofa_entry_df = validate_and_intersect_views(
    rna_df=log_df,
    csf_df=csfp_clean,
    plasma_df=plap_clean,
    metadata_df=meta.loc[meta.study=="PPMI"],
    output_dir="../ppmi_mofa_inputs"
)

### Train MOFA

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from train_mofa import train_mofa_with_seeds

In [ ]:
train_mofa_with_seeds("../ppmi_mofa_inputs/ppmi_mofa_input_long.csv", "../ppmi_mofa_outputs", n_seeds=5, n_factors=10)

# PDBP

## Preprocessing

### Prepare metadata

In [ ]:
# Checkpoint
meta_pdbp = pd.read_csv('../data/metadata_m0_PDBP.csv')

### Prepare expression data

In [ ]:
# Run in terminal
! python3 scripts/preprocess_pdbp.py --input_dir data/pdbp/raw/ --output_dir pdbp_mofa_inputs/ --cohort PDBP --model_path ppmi_mofa_outputs/model_seed_1.hdf5

# OUTPUT:
#Starting PDBP preprocessing for cohort: PDBP
#Processing RNA_seq from data/pdbp/raw/rna_seq.csv...
#Saved aligned RNA_seq matrix to pdbp_mofa_inputs/pdbp_rna_seq_aligned.csv (Shape: (5000, 1248))
#Processing CSF_Proteomics from data/pdbp/raw/csf_proteomics.csv...
#Saved aligned CSF_Proteomics matrix to pdbp_mofa_inputs/pdbp_csf_proteomics_aligned.csv (Shape: (249, 107))
#Processing Plasma_Proteomics from data/pdbp/raw/plasma_proteomics.csv...Saved aligned Plasma_Proteomics matrix to pdbp_mofa_inputs/pdbp_plasma_proteomics_aligned.csv (Shape: (202, 107))
#
#Preprocessing complete.

In [ ]:
# Verify that data was processed correctly
import h5py
import pandas as pd
import numpy as np

model_path = "../ppmi_mofa_outputs/model_seed_1.hdf5"
views = {
    "RNA_seq": "../pdbp_mofa_inputs/pdbp_rna_seq_aligned.csv",
    "CSF_Proteomics": "../pdbp_mofa_inputs/pdbp_csf_proteomics_aligned.csv",
    "Plasma_Proteomics": "../pdbp_mofa_inputs/pdbp_plasma_proteomics_aligned.csv"
}

print("=== 1. Feature Order & Count Alignment ===")
with h5py.File(model_path, 'r') as f:
    for view, file_path in views.items():
        # Get model features
        model_feats = [x.decode('utf-8') if isinstance(x, bytes) else str(x) 
                       for x in f['features'][view][:]]
        
        # Load aligned matrix
        df = pd.read_csv(file_path, index_col=0)
        file_feats = list(df.index)
        
        # Check alignment
        is_exact_match = (model_feats == file_feats)
        print(f"[{view}] Model features: {len(model_feats)} | File features: {len(file_feats)} | Perfect Match: {is_exact_match}")

print("\n=== 2. Missing Value (NaN) & Scaling Check ===")
for view, file_path in views.items():
    df = pd.read_csv(file_path, index_col=0)
    
    nan_pct = (df.isna().sum().sum() / df.size) * 100
    mean_val = df.mean().mean()
    min_val, max_val = df.min().min(), df.max().max()
    
    print(f"[{view}] Missing values: {nan_pct:.2f}% | Mean: {mean_val:.3f} | Min: {min_val:.3f} | Max: {max_val:.3f}")

print("\n=== 3. Sample ID Consistency Check ===")
rnaseq_samples = set(pd.read_csv(views["RNA_seq"], index_col=0).columns)
csf_samples = set(pd.read_csv(views["CSF_Proteomics"], index_col=0).columns)
plasma_samples = set(pd.read_csv(views["Plasma_Proteomics"], index_col=0).columns)

print(f"Total RNA-seq samples: {len(rnaseq_samples)}")
print(f"Total CSF samples: {len(csf_samples)}")
print(f"Total Plasma samples: {len(plasma_samples)}")
print(f"Samples in CSF & Plasma: {len(csf_samples.intersection(plasma_samples))}")
print(f"Samples in RNA-seq & CSF: {len(rnaseq_samples.intersection(csf_samples))}")